DEPENDICIES

In [44]:
pip install -r requirements.txt -q


Note: you may need to restart the kernel to use updated packages.


In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
import timm
from sklearn.model_selection import train_test_split


In [46]:
#for nvidia GPU    NOT FULLY WORKING YET ALSO NEED TO ADD APPLE CHIP

import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using device: {device}')

2.10.0+cu128
12.8
True
NVIDIA GeForce RTX 5080
using device: cuda


In [ ]:
batchsize = 8
workers = 0   # must be 0 in Jupyter — multiprocessing can't pickle notebook-defined classes

arcs = 64
arcm = 0.5
K = 3      # sub-centers per jaguar: 1 = standard ArcFace, 3 = Sub-Center ArcFace


GATHER DATASETS

In [48]:
testdf = pd.read_csv('data/test.csv')
traindf = pd.read_csv('data/train.csv')

traindir = Path('data/train')
testdir = Path('data/test')


EDA

In [49]:
print('train columns: ',traindf.columns.to_list())
print(traindf.head())
print('+++++++++++++++++++++++++++++++++++++++++++++')
print(f'Total training images: {len(traindf)}')

print(f'unique jaguars: {traindf['ground_truth'].nunique()}')
print('+++++++++++++++++++++++++++++++++++++++++++++')
print('class blanace:')
counts = traindf['ground_truth'].value_counts()
print(f'Images per Jaguar:')
print(f'min: {counts.min()}')
print(f'max: {counts.max()}')
print(f'mean: {counts.mean()}')

print('\ntest structure:')
print(testdf.head())
print(f'total pairs: ')
print(len(testdf))

train columns:  ['filename', 'ground_truth']
         filename ground_truth
0  train_0001.png        Abril
1  train_0002.png        Abril
2  train_0003.png        Abril
3  train_0004.png       Akaloi
4  train_0005.png       Akaloi
+++++++++++++++++++++++++++++++++++++++++++++
Total training images: 1895
unique jaguars: 31
+++++++++++++++++++++++++++++++++++++++++++++
class blanace:
Images per Jaguar:
min: 13
max: 183
mean: 61.12903225806452

test structure:
   row_id    query_image  gallery_image
0       0  test_0001.png  test_0002.png
1       1  test_0001.png  test_0003.png
2       2  test_0001.png  test_0004.png
3       3  test_0001.png  test_0005.png
4       4  test_0001.png  test_0006.png
total pairs: 
137270


setting up transformers

In [50]:
traintransform = transforms.Compose([
    transforms.RandomResizedCrop(384, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
valtransform = transforms.Compose([
    transforms.Resize(384),
    transforms.CenterCrop(384),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


encodes the names of each jaguar to ints alphabetically
'abril' = 0
'akaloi' = 1
so on 

In [51]:
class jaguardataset(Dataset):
    def __init__ (self, df: pd.DataFrame, dir: Path, transform=None):
        self.df = df.reset_index(drop=True)
        self.dir = dir
        self.transform = transform
        idx = sorted(self.df['ground_truth'].unique())
        self.labelmap = {v: i for i, v in enumerate(idx)}
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.dir / row['filename']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = self.labelmap[row['ground_truth']]
        return img, label
    
from sklearn.model_selection import train_test_split

trainsplit, valsplit = train_test_split(
    traindf, test_size=0.2,
    stratify=traindf['ground_truth'],
    random_state=99
)

trainds = jaguardataset(trainsplit, traindir, traintransform)
valds = jaguardataset(valsplit, traindir, valtransform)


weighted sampler because some jaguars have over 100 images when other only have a few

In [52]:
from torch.utils.data import WeightedRandomSampler

counts = trainsplit['ground_truth'].value_counts()
classweight = 1 / counts
sampleweight = trainsplit['ground_truth'].map(classweight).values
sampler = WeightedRandomSampler(
    weights=sampleweight,
    num_samples=len(sampleweight),
    replacement=True
)


LOADER

In [53]:
pin = device.type == 'cuda'  # pin_memory only works on CUDA, not MPS or CPU

Trainloader = DataLoader(trainds, batch_size=batchsize, sampler=sampler, num_workers=workers, pin_memory=pin)
valloader   = DataLoader(valds,   batch_size=batchsize, shuffle=False,   num_workers=workers, pin_memory=pin)


In [54]:
# ── MODEL ARCHITECTURE ───────────────────────────────────────────────────────
# Pipeline: MegaDescriptor (Swin-L backbone) → CBAM attention → Sub-Center ArcFace head
#
# CBAM forces the model to focus on jaguar spots/flanks rather than the background.
# Sub-Center ArcFace gives each jaguar K cluster centers so left-flank and
# right-flank views (which have different spot patterns) can both be captured.
# K=1 reduces to standard ArcFace; K=3 is the sub-center variant.

# --- Channel + Spatial Attention (CBAM) ---
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = self.fc(self.avg_pool(x).flatten(1))
        mx  = self.fc(self.max_pool(x).flatten(1))
        return x * self.sigmoid(avg + mx).unsqueeze(-1).unsqueeze(-1)

print('channel done')
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.amax(dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

print('spacial attention done')
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel = ChannelAttention(channels, reduction)
        self.spatial = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.spatial(self.channel(x))

print('cbam done')
# --- Sub-Center ArcFace ---
# K sub-centers per class so multi-view images (left/right flank) cluster correctly.
# When K=1 this is equivalent to standard ArcFace.
class SubCenterArcFace(nn.Module):
    def __init__(self, in_features, num_classes, K=3, s=64.0, m=0.5):
        super().__init__()
        self.K           = K
        self.s           = s
        self.m           = m
        self.num_classes = num_classes
        self.weight      = nn.Parameter(torch.empty(num_classes * K, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, features, labels, margin=None):
        m   = margin if margin is not None else self.m
        cos = F.linear(F.normalize(features), F.normalize(self.weight))  # (B, C*K)
        cos = cos.view(-1, self.num_classes, self.K).amax(dim=2)          # (B, C) best sub-center
        theta   = cos.clamp(-1 + 1e-7, 1 - 1e-7).acos()
        one_hot = torch.zeros_like(cos).scatter_(1, labels.unsqueeze(1), 1.0)
        logits  = cos * (1 - one_hot) + torch.cos(theta + m) * one_hot
        return logits * self.s

print('arcface done')
# --- Full Model ---
class JaguarIDModel(nn.Module):
    """MegaDescriptor-L backbone + CBAM spatial attention + Sub-Center ArcFace."""

    def __init__(self, num_classes, embed_dim=512, K=3, s=64.0, m=0.5):
        super().__init__()
        self.backbone = timm.create_model(
            'hf-hub:BVRA/MegaDescriptor-L-384',
            pretrained=True, num_classes=0, global_pool=''
        )
        feat_dim   = self.backbone.num_features          # 1536 for Swin-L
        self.cbam  = CBAM(feat_dim)
        self.pool  = nn.AdaptiveAvgPool2d(1)
        self.neck  = nn.Sequential(nn.Linear(feat_dim, embed_dim), nn.BatchNorm1d(embed_dim))
        self.head  = SubCenterArcFace(embed_dim, num_classes, K=K, s=s, m=m)

    def _to_nchw(self, x):
        """Normalize Swin feature map to (B, C, H, W) regardless of timm version."""
        if x.dim() == 4 and x.shape[1] != x.shape[3]:   # (B, H, W, C) NHWC
            return x.permute(0, 3, 1, 2).contiguous()
        if x.dim() == 3:                                  # (B, L, C) — assume square spatial
            B, L, C = x.shape
            H = W = int(L ** 0.5)
            return x.view(B, H, W, C).permute(0, 3, 1, 2).contiguous()
        return x                                          # already (B, C, H, W)

    def embed(self, x):
        x = self._to_nchw(self.backbone.forward_features(x))
        x = self.pool(self.cbam(x)).flatten(1)
        return F.normalize(self.neck(x), p=2, dim=1)

    def forward(self, x, labels, margin=None):
        return self.head(self.embed(x), labels, margin)

num_classes = len(trainds.labelmap)
model       = JaguarIDModel(num_classes=num_classes, embed_dim=512, K=K, s=arcs, m=arcm).to(device)
print(f'Model ready — K={K} | {num_classes} jaguar classes | {sum(p.numel() for p in model.parameters()):,} params')


channel done
spacial attention done
cbam done
arcface done
Model ready — K=5 | 31 jaguar classes | 196,360,854 params


In [55]:
# ── TRAINING ─────────────────────────────────────────────────────────────────
# Dynamic margin: starts small (embeddings are still random) and ramps to arcm
# so the model can orient before the ArcFace penalty gets tight.

from sklearn.metrics import roc_auc_score

def dynamic_margin(epoch, total, m_start=0.1, m_end=0.5):
    return m_start + (m_end - m_start) * min(epoch / total, 1.0)


epochs = 20

# Layer-wise learning rates: backbone gets 10x lower LR to preserve pretrained weights
backbone_params = list(model.backbone.parameters()) + list(model.cbam.parameters())
head_params     = list(model.pool.parameters()) + list(model.neck.parameters()) + list(model.head.parameters())

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5},
    {'params': head_params,     'lr': 3e-4},
], weight_decay=1e-4)

# Warmup for 2 epochs, then cosine decay
warmup_epochs    = 2
warmup_scheduler = optim.lr_scheduler.LambdaLR(optimizer, lambda epoch: (epoch + 1) / warmup_epochs if epoch < warmup_epochs else 1.0)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)
scheduler        = optim.lr_scheduler.SequentialLR(optimizer, [warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

for epoch in range(epochs):
    model.train()
    margin  = dynamic_margin(epoch, epochs, m_start=0.1, m_end=arcm)
    tr_loss = 0

    for imgs, labels in tqdm(Trainloader, desc=f'Train {epoch+1}/{epochs}', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs, labels, margin=margin), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        tr_loss += loss.item()

    scheduler.step()

    # Validation AUC — embed all val images, score same/different pairs
    model.eval()
    val_embs, val_lbls = [], []
    with torch.no_grad():
        for imgs, labels in valloader:
            val_embs.append(model.embed(imgs.to(device)).cpu())
            val_lbls.append(labels)
    val_embs = torch.cat(val_embs)
    val_lbls = torch.cat(val_lbls)

    # Build all pairs (n~380, so ~72k pairs — fast)
    sims, targets = [], []
    for i in range(len(val_embs)):
        for j in range(i + 1, len(val_embs)):
            sims.append(F.cosine_similarity(val_embs[i].unsqueeze(0), val_embs[j].unsqueeze(0)).item())
            targets.append(int(val_lbls[i].item() == val_lbls[j].item()))

    val_auc = roc_auc_score(targets, sims)
    print(f'Epoch {epoch+1:02d} | K={K} | margin={margin:.3f} | loss={tr_loss/len(Trainloader):.4f} | val_auc={val_auc:.4f}')

torch.save(model.state_dict(), f'jaguar_model_K{K}.pth')
print(f'Saved jaguar_model_K{K}.pth')


Epoch 01 | K=5 | margin=0.100 | loss=7.2895 | val_auc=0.6858


Epoch 02 | K=5 | margin=0.120 | loss=4.8565 | val_auc=0.7319


Epoch 03 | K=5 | margin=0.140 | loss=3.6022 | val_auc=0.7701


Epoch 04 | K=5 | margin=0.160 | loss=3.1380 | val_auc=0.7974


Epoch 05 | K=5 | margin=0.180 | loss=2.8546 | val_auc=0.8129


...

Epoch 16 | K=5 | margin=0.400 | loss=2.3872 | val_auc=0.9012


Epoch 17 | K=5 | margin=0.420 | loss=2.3946 | val_auc=0.9040


Epoch 18 | K=5 | margin=0.440 | loss=2.5213 | val_auc=0.8992


Epoch 19 | K=5 | margin=0.460 | loss=2.9840 | val_auc=0.8942


Epoch 20 | K=5 | margin=0.480 | loss=3.1599 | val_auc=0.9033
Saved jaguar_model_K5.pth


In [56]:
# ── INFERENCE ────────────────────────────────────────────────────────────────
# Embed every test image once, then compute cosine similarity for each pair.

class TestImgDataset(Dataset):
    def __init__(self, filenames, img_dir, transform):
        self.files     = sorted(filenames)
        self.dir       = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img   = Image.open(self.dir / fname).convert('RGB')
        return self.transform(img), fname


all_test_imgs  = sorted(set(testdf['query_image'].tolist() + testdf['gallery_image'].tolist()))
test_imgds     = TestImgDataset(all_test_imgs, testdir, valtransform)
test_imgloader = DataLoader(test_imgds, batch_size=batchsize, shuffle=False, num_workers=workers)

model.eval()
embeddings = {}
with torch.no_grad():
    for imgs, fnames in tqdm(test_imgloader, desc='Embedding test images'):
        embs = model.embed(imgs.to(device))
        for fname, emb in zip(fnames, embs.cpu()):
            embeddings[fname] = emb

# Vectorized cosine similarity for all pairs
q_embs = torch.stack([embeddings[r] for r in testdf['query_image']])
g_embs = torch.stack([embeddings[r] for r in testdf['gallery_image']])
sims   = F.cosine_similarity(q_embs, g_embs)       # range [-1, 1]
similarities = ((sims + 1) / 2).numpy()             # rescale to [0, 1]

print(f'Similarities — min={similarities.min():.4f}  max={similarities.max():.4f}  mean={similarities.mean():.4f}')



Embedding test images: 100%|██████████| 47/47 [00:53<00:00,  1.14s/it]


Similarities — min=0.3029  max=0.9981  mean=0.5084


CREATE SUBMISSION

In [57]:
submission = pd.DataFrame({
    'row_id': testdf['row_id'],
    'similarity': similarities
})

# Validate
assert len(submission) == 137270
assert submission['row_id'].tolist() == list(range(137270))
assert (submission['similarity'] >= 0).all()
assert (submission['similarity'] <= 1).all()

print("\nSubmission statistics:")
print(submission['similarity'].describe())


Submission statistics:
count    137270.000000
mean          0.508413
std           0.073953
min           0.302878
25%           0.462321
50%           0.499697
75%           0.541728
max           0.998133
Name: similarity, dtype: float64


VALIDATION AND SUBMISSION

In [58]:
validate_submission('my_submission.csv')

# Save submission
submission.to_csv('my_submission.csv', index=False)
print("\nSubmission saved: my_submission.csv")
print(f"File size: {Path('my_submission.csv').stat().st_size / 1024 / 1024:.2f} MB")

NameError: name 'validate_submission' is not defined